# 攻防规则首轮仿真：23×12 米

同一块 75×75 厘米靶架的五孔全部击落，攻方获胜；120 秒未完成则守方胜。坐标、孔径、箭速、能力映射和复活耗时尚未实测。掩体暂按 1×0.6×1 米解释，需确认。结果只代表这组假设与脚本策略，不是现实胜率。


In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT_DIR = Path("/content/archery-tag-simulate")
REPO_URL = "https://github.com/Echoudexigaigu/archery-tag-simulate.git"
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    if not (PROJECT_DIR / ".git").exists():
        raise RuntimeError("项目目录没有 Git 历史，请先备份，再使用一个新的 Colab 运行时。")
    remote = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "remote", "get-url", "origin"], text=True).strip()
    if remote.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("已有目录属于其他仓库；请使用新的运行时。")
    dirty = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True).strip()
    if dirty:
        print("保留已有未提交改动，不自动更新。请先保存，再决定如何更新。")
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if not (PROJECT_DIR / "src/archery_tag_balance/environment.py").is_file():
    raise RuntimeError("已有目录还是旧骨架。保存改动后更新仓库，或在新运行时重新运行。")
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True).strip())


## 配置与依赖

仿真底层只使用标准库。绘图使用 Matplotlib，下面仅在版本不匹配时安装绘图依赖。修改配置副本即可改变靶位或老手人数。


In [ ]:
import importlib.metadata
try:
    plotting_version = importlib.metadata.version("matplotlib")
except importlib.metadata.PackageNotFoundError:
    plotting_version = None
if plotting_version != "3.10.8":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
import json
from copy import deepcopy
config = json.loads((PROJECT_DIR / "configs/attack_defense_23x12.json").read_text())
print(json.dumps(config["provenance"], ensure_ascii=False, indent=2))
# 可修改：config["rack"]["position_xyz_m"] = [20.0, 6.0, 1.1]
# 可修改：config["teams"]["attack"]["veterans"] = 3
# 可修改：config["teams"]["defense"]["veterans"] = 1


In [ ]:
import matplotlib.pyplot as plt
from archery_tag_balance.visualization import plot_layout, plot_resources
fig = plot_layout(config)
plt.show()


## 单局试跑

SEED 决定熟练度抽样和射击误差。末尾的 run_dir 保存当前配置和结果；如需跨运行时保留，另外备份至 Drive。


In [ ]:
from archery_tag_balance import SimulationEngine
from archery_tag_balance.experiments import save_json, source_digest
SEED = 1000
summary, env = SimulationEngine(config).episode(SEED, record=True, check_invariants=True)
print(json.dumps({k:v for k,v in summary.items() if k not in ("eta", "resource_curve")}, ensure_ascii=False, indent=2))
run_dir = PROJECT_DIR / "runs" / f"demo_{SEED}"
save_json(run_dir / "config.json", config)
save_json(run_dir / "episode.json", {"summary":summary,"trace":env.trace,"events":env.events,"source_sha256":source_digest()})
fig = plot_resources(summary["resource_curve"])
plt.show()


## 检查比赛中某一时刻

改变 frame_index 可以查看不同时间。靶架正视图区分五个孔；俯视图不能区分上下孔。


In [ ]:
frame_index = min(30, len(env.trace)-1)
fig = plot_layout(config, env=env, frame=env.trace[frame_index])
plt.show()


## 批量比较阵容

先跑每类 20 局检查流程。提高 N 才能降低 Monte Carlo 误差；统计区间不包含物理模型或脚本偏差。


In [ ]:
N = 20
comparisons = []
for av, dv in [(2,2),(3,1),(1,3)]:
    scenario = deepcopy(config)
    scenario["teams"]["attack"]["veterans"] = av
    scenario["teams"]["defense"]["veterans"] = dv
    stats, episodes = SimulationEngine(scenario).batch(range(2000,2000+N))
    comparisons.append({"attack_veterans":av,"defense_veterans":dv,"summary":stats,"config":scenario,"episodes":episodes})
    print(f"老手 {av} vs {dv}: 攻方胜率 {stats["attack_win_rate"]:.1%}; 95%区间 {stats["wilson_95"]}")
save_json(run_dir / "roster_comparison.json", comparisons)


## 完整实验与已提交报告

完整扫描脚本：scripts/run_initial_experiments.py。运行前确认剩余会话时间，并将输出指向可备份位置。先阅读 reports/first_simulation/README.md 中的假设、样本量与限制。当前没有训练 RL 策略。
